#### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import KMRF class
from kmrf import KMRF
from KMRF_training_config import *

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")
print(f"  Pandas version: {pd.__version__}")
print(f"  NumPy version: {np.__version__}")

✓ Libraries imported successfully
  Pandas version: 2.3.3
  NumPy version: 2.3.4


## 1. Configure KMRF Model

In [2]:
# Create DataFrame of asset names
asset_names_df = pd.DataFrame({
    'Investment Universe ETFs': get_assets_by_class('universe') + ['']*7,
    'US Equity ETFs': get_assets_by_class('us_equity'),
    'Commodities': get_assets_by_class('commodity') + ['']*5,
    'International Equity ETFs': get_assets_by_class('int_equity') + ['']*11,

})

from pandas import option_context
with option_context('display.max_colwidth', None):
    display(asset_names_df)

,Investment Universe ETFs,US Equity ETFs,Commodities,International Equity ETFs
0,IVV - iShares Core S&P 500 ETF,SPDR S&P 500 ETF,Gold Futures,Vanguard Total International Stock ETF
1,IJH - iShares Core S&P Mid-Cap ETF,Invesco QQQ Trust,Wheat Futures,Vanguard FTSE Developed Markets ETF
2,IWM - iShares Russell 2000 ETF,iShares Russell 2000 ETF,Corn Futures,Vanguard FTSE Emerging Markets ETF
3,EFA - iShares MSCI EAFE ETF,SPDR Dow Jones Industrial Average ETF,Copper,Vanguard FTSE Europe ETF
4,EEM - iShares MSCI Emerging Markets ETF,Energy Select Sector SPDR,Sugar,Vanguard FTSE Pacific ETF
5,AGG - iShares Core U.S. Aggregate Bond ETF,Financial Select Sector SPDR,Silver Futures,iShares China Large-Cap ETF
6,SPTL - SPDR Portfolio Long Term Treasury ETF,Utilities Select Sector SPDR,US Dollar,iShares MSCI Japan ETF
7,HYG - iShares iBoxx $ High Yield Corporate Bond ETF,Industrial Select Sector SPDR,Soybean Futures,iShares MSCI India ETF
8,SPBO - SPDR Portfolio Corporate Bond ETF,Health Care Select Sector SPDR,Lumber Futures,
9,IYR - iShares U.S. Real Estate ETF,Technology Select Sector SPDR,Live Cattle Futures,


In [3]:
# Configuration for multi-horizon prediction
TRAINING_CONFIG = KMRF_Training_Config(
    asset_name='SPDR S&P 500 ETF',
    classification_type='original',  # or 'adapted' for 3-class
    use_data_type='master',
    end_date='20251007',
    feature_window_size=1,
    feature_asset_classes=[],
    cross_asset_specific=[],
    use_boruta_selection=False,
    use_consensus_selection=False
)

# Multi-horizon specific settings
MAX_HORIZON = 21  # Predict up to 21 days forward
ASSET_NAME = 'SPDR S&P 500 ETF'
ASSET_CLASS = 'us_equity'
KAMAMSR_END_DATE = '20251007'


## 2. Initialize KMRF Model

In [4]:
# Initialize KMRF model
model = KMRF(
    asset_class=ASSET_CLASS,
    asset_name=ASSET_NAME,
    classification_type='original',
    end_date=KAMAMSR_END_DATE,
    use_data_type='master',
    feature_window_size=1,  
    feature_asset_classes=[],
    cross_asset_specific=[],
    xgb_params=TRAINING_CONFIG.get_xgb_params(),
    use_boruta_selection=False,
    use_consensus_selection=False,
)

print("\n" + "="*80)
print("KMRF MODEL INITIALIZED")
print("="*80)

KMRF model initialized
  Asset: SPDR S&P 500 ETF
  Asset class: us_equity
  Classification type: original
  Training end date: 20251007
  Test end date: All available data
  Data type: master
  Data path: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/data/master_df.csv
  KAMA+MSR model directory: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/saved_models/KAMA_MSR/us_equity/20251007
  Validation/Test split: Will be calculated after loading data
  Random seed: 1010
  Feature window size: 1 days
  Feature asset classes: []
  Feature selection: Boruta=False, Consensus=False
  Custom XGB parameters: {'n_estimators': 220, 'max_depth': 13, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.25, 'min_child_weight': 95, 'gamma': 0.045, 'random_state': 1010, 'n_jobs': -1, 'tree_method': 'hist', 'enable_categorical': False}

KMRF MODEL INITIALIZED


## 3. Run Training Pipeline

In [5]:
model.pipeline(optimize=False)


KMRF PIPELINE FOR SPDR S&P 500 ETF
Asset Class: us_equity
Classification Type: original
Data Type: master
Feature Asset Classes: []
Cross-Asset Specifics: []
    ^ if empty, uses all assets in 'Feature Asset Classes'
Feature Window Size: 1
Use Boruta Selection: False
Use Consensus Selection: False

[Step 1/10] Loading Data...

Loading data from: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/data/master_df.csv
Loaded data for: SPDR S&P 500 ETF
  Rows: 8246
  Columns: 94
  Date range: 1993-02-01 00:00:00 to 2025-10-31 00:00:00

[Step 2/10] Computing Features...

Extracting pre-computed features for SPDR S&P 500 ETF...
  Features shape: (8246, 94)

[Step 3/10] Loading KAMA+MSR Labels...

LOADING KAMA+MSR LABELS FOR SPDR S&P 500 ETF
Model directory: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/saved_models/KAMA_MSR/us_equity/20251007
Loading from: SPDR S&P 500 ETF_KAMA-MSR_4-regimes.pkl
✓ Loaded labels for: SPDR S&P 500 ETF
  Label date ran

In [6]:
model.X_train.tail()

,open_lag1d,high_lag1d,low_lag1d,close_lag1d,volume_lag1d,vwap_lag1d,log_ret_overnight,log_ret_intraday_lag1d,log_ret_lag1d,log_ret_5d_lag1d,...,tsfresh_volume_lag1d__sum_values,tsfresh_volume_lag1d__median,tsfresh_volume_lag1d__mean,tsfresh_volume_lag1d__length,tsfresh_volume_lag1d__standard_deviation,tsfresh_volume_lag1d__variance,tsfresh_volume_lag1d__root_mean_square,tsfresh_volume_lag1d__maximum,tsfresh_volume_lag1d__absolute_maximum,tsfresh_volume_lag1d__minimum
date,,,,,,,,,,,,,,,,,,,,,
2025-10-01,662.93,666.65,661.61,666.18,94151106.0,664.130,-0.004529,0.004891,0.003760,0.004468,...,1.520061e+09,71800517.5,76003042.40,20.0,1.159195e+07,1.343733e+14,7.688196e+07,101952244.0,101952244.0,61169000.0
2025-10-02,663.17,669.37,663.06,668.45,72545400.0,666.215,0.002988,0.007930,0.003402,0.011056,...,1.532228e+09,71800517.5,76611420.45,20.0,1.219354e+07,1.486825e+14,7.757572e+07,101952244.0,101952244.0,61169000.0
2025-10-03,670.45,670.57,666.78,669.22,56896000.0,668.675,0.001150,-0.001836,0.001151,0.016832,...,1.533953e+09,72662767.5,76697645.45,20.0,1.215834e+07,1.478252e+14,7.765535e+07,101952244.0,101952244.0,61169000.0
2025-10-06,669.99,672.68,668.16,669.21,70494400.0,670.420,0.003595,-0.001165,-0.000015,0.011104,...,1.525630e+09,72662767.5,76281484.05,20.0,1.267555e+07,1.606695e+14,7.732745e+07,101952244.0,101952244.0,56896000.0
2025-10-07,671.62,672.51,669.46,671.61,54623300.0,670.985,0.001384,-0.000015,0.003580,0.011878,...,1.510945e+09,71519900.0,75547257.30,20.0,1.256371e+07,1.578467e+14,7.658482e+07,101952244.0,101952244.0,56896000.0


In [7]:
model.y_train.tail()

date
2025-10-01    0
2025-10-02    0
2025-10-03    0
2025-10-06    0
2025-10-07    0
dtype: Int64

## 4. Generate & Visualize Predictions

In [8]:
model.predict_all_oos()


Generating predictions for all out-of-sample data...
  Input shape: (18, 94)
  Date range: 2025-10-08 00:00:00 to 2025-10-31 00:00:00
✓ Generated predictions for all OOS data: (18, 4)
  Date range: 2025-10-08 00:00:00 to 2025-10-31 00:00:00


,P(LV_Bull),P(LV_Bear),P(HV_Bull),P(HV_Bear)
date,,,,
2025-10-08,0.998752,0.000360,0.000733,0.000156
2025-10-09,0.998318,0.000615,0.000861,0.000206
2025-10-10,0.998770,0.000362,0.000716,0.000152
2025-10-13,0.995836,0.000371,0.003343,0.000451
2025-10-14,0.997436,0.000487,0.001724,0.000353
2025-10-15,0.997628,0.000479,0.001634,0.000258
2025-10-16,0.998803,0.000263,0.000790,0.000144
2025-10-17,0.997158,0.000459,0.002080,0.000304
2025-10-20,0.997758,0.000503,0.001414,0.000325


## 5. Save Model

In [26]:
path = f'saved_models/KMRF_new/{model.classification_type}/{model.asset_class}/'
path += f'{model.asset_name.replace(" ", "_")}_KMRF_model.pkl'
model.save_model(path)


Saving model to: saved_models/KMRF_new/original/us_equity/SPDR_S&P_500_ETF_KMRF_model.pkl
  Asset: SPDR S&P 500 ETF
  Asset class: us_equity
  Classification type: original
  Features: all
  Training samples: 7490
✓ Model saved successfully


PosixPath('saved_models/KMRF_new/original/us_equity/SPDR_S&P_500_ETF_KMRF_model.pkl')